# 05. Silver: embed every review

Turns each review that has text into a 384-number vector and stores it in an Iceberg table.

## What to look at

- **dim = 384** and **norm = 1.0** for every row (unit vectors, so cosine similarity equals dot product).
- Reviews with no text are not in Silver.
- Iceberg stores the vector as a plain `array<float>`; it has no vector type.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
run_script("05_silver_embeddings.py")

In [ ]:
spark = get_spark("nb05")
sql(spark, f"""SELECT review_text, order_status, review_score, size(embedding) AS dim,
                      round(sqrt(aggregate(embedding, CAST(0 AS DOUBLE), (a, x) -> a + x * x)), 4) AS norm,
                      slice(embedding, 1, 4) AS first_4_numbers
               FROM {T_SILVER} LIMIT 8""")

In [ ]:
sql(spark, f"""SELECT count(*) AS rows, min(size(embedding)) AS min_dim, max(size(embedding)) AS max_dim FROM {T_SILVER}""")

In [ ]:
bronze_text = spark.table(T_BRONZE).where("has_review_text").count()
print("bronze rows with text:", bronze_text, "| silver rows:", spark.table(T_SILVER).count())

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

New folder `silver/review_embeddings/` with **data/** (Parquet; bigger files than Bronze because each row carries 384 floats) and **metadata/**.